> 在线讲义：[第 16 章 · ALiBi 与 NoPE：不加位置编码行不行](https://wikieden.github.io/beyond-transformer/lessons/16-alibi-nope.html) · 本书《超越 Transformer》

# 第 16 章 · ALiBi 与 NoPE：不加位置编码行不行

配套讲义：[在线版](https://wikieden.github.io/beyond-transformer/lessons/16-alibi-nope.html) · 仓库里 `beyond-transformer/notes/lessons/16-alibi-nope.md`（附录最后一节"代码对照"）

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的 Python 写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。只用 NumPy，不需要 GPU。练习题不在这里算，先在讲义里自己做。

In [ ]:
import numpy as np

np.set_printoptions(precision=4, suppress=True)  # 打印保留 4 位小数，不用科学计数法
print("numpy", np.__version__)

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。

| 讲义 | 公式 | 写法 | 跑出来 |
|---|---|---|---|
| 第一步 偏置与权重 | −m·距离 | `alibi_bias(5, m)`、`alibi_weights(m)` | m = 0.5 时最后一个位置 0.4087 |
| 第二步 斜率 | 2^{−8k/H} | `slopes(8)`、`slopes(16)` | 1/2 … 1/256 |
| 第三步 NoPE | 1/(i+1) | `nope_position(8)` | 1、0.5、0.3333 … |

NoPE 什么都不加，只靠因果 mask：第一层均匀地看前面所有词，就能算出“我是第几个”。
a1 的 no_pos_emb 消融和五种位置编码的外推对比（里程碑 T-M3）留给自己做。

## 1. 先把要用的东西定义好

这几格只是定义，运行后没有输出也正常。

**函数 `slopes`**：ALiBi §3（L331）：8 个头时斜率是 1/2¹、1/2²、…、1/2⁸；一般 H 个头是 2^{−8k/H}。

In [ ]:
def slopes(h: int):
    """ALiBi §3（L331）：8 个头时斜率是 1/2¹、1/2²、…、1/2⁸；一般 H 个头是 2^{−8k/H}。"""
    return [2 ** (-8 * (k + 1) / h) for k in range(h)]

**函数 `alibi_bias`**：位置 i 看位置 j（j ≤ i）：分数加 −m·(i − j)。

In [ ]:
def alibi_bias(n: int, m: float):
    """位置 i 看位置 j（j ≤ i）：分数加 −m·(i − j)。"""
    i, j = np.meshgrid(np.arange(n), np.arange(n), indexing="ij")
    b = -m * (i - j).astype(float)
    b[j > i] = -np.inf
    return b

**函数 `softmax_row`**：小工具。

In [ ]:
def softmax_row(s):
    s = s - s[np.isfinite(s)].max()
    e = np.where(np.isfinite(s), np.exp(s), 0.0)
    return e / e.sum()

In [ ]:
SCORES = np.array([1.0, 0.5, 2.0, 0.8, 1.2])   # 最后一个位置对前 5 个位置（含自己）的原始分数

**函数 `alibi_weights`**：小工具。

In [ ]:
def alibi_weights(m: float, scores=SCORES):
    n = len(scores)
    return softmax_row(scores + alibi_bias(n, m)[n - 1])

**函数 `nope_position`**：第一层一个头：所有分数相同（均匀看），value 只有句首 token 是 1。因果 mask 下位置 i 看 i + 1 个词，

In [ ]:
def nope_position(n: int = 8):
    """第一层一个头：所有分数相同（均匀看），value 只有句首 token 是 1。因果 mask 下位置 i 看 i + 1 个词，
    输出 = 1/(i + 1)，单调对应位置，后面的 MLP 就能读出“我是第几个”（NoPE 论文定理 1 的思路）。"""
    v = np.zeros(n)
    v[0] = 1.0
    return [float(np.mean(v[: i + 1])) for i in range(n)]

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("第一步 ALiBi 的斜率")
print("  8 个头：" + "、".join(f"1/{round(1 / s)}" for s in slopes(8)))
print("  16 个头的前 4 个：" + "、".join(f"{s:.4f}" for s in slopes(16)[:4]))
print("\n第二步 分数减一个斜坡")
print(f"  最后一个位置的原始分数 {SCORES.tolist()}（距离 4、3、2、1、0）")
for m in (0.0, 0.5, 1 / 256):
    print(f"  斜率 {m:.4f}：偏置 {(np.round(alibi_bias(5, m)[4], 4) + 0.0).tolist()}，权重 {np.round(alibi_weights(m), 4).tolist()}")

In [ ]:
print("\n第三步 NoPE：因果 mask 本身就带着位置")
print(f"  均匀注意力 + 句首标记：各位置的输出 {np.round(nope_position(), 4).tolist()}")

## 3. 代码里才看得到的

- **偏置和 mask 合在一起**：被遮的位置直接放 −∞，softmax 时它们的权重自然是 0；实现时 ALiBi 偏置通常和因果 mask 拼成同一个张量传进注意力。
- **NoPE 的“位置”是第一层算出来的**：代码里的均匀注意力和句首 value 都是手写的；真实模型要从数据里学出这样的头。

## 4. 自己改一改

- 把 `SCORES` 改成递增的 [0, 1, 2, 3, 4]，斜率 0.5 时权重还偏向最近吗？
- 给 `nope_position` 换成双向（每个位置都看全部 n 个），输出变成什么？

在下面的空格子里写，或者直接改上面的格子。

In [ ]:
# 在这里写你的代码

## 口诀

> 离讲台越远听得越轻：分数减斜率乘距离，不改 q k，也不加在词上。
> 斜率不学，几何数列，从二分之一到二百五十六分之一；只看距离，长了也不怕。
> 不加位置靠 mask：均匀看前面，句首当一，输出一除以 i 加一，就知道排第几；双向就做不到。